# 05 — Spectral Analysis of Small-World Networks

## Motivation

In the previous notebooks we observed empirically that the Watts-Strogatz
network mixes much faster than the ring lattice. But *why*?

This notebook develops the mathematical tools that give a **rigorous answer**.
The central object is the **graph Laplacian**: a matrix built from the network
whose eigenvalues directly control how fast the random walk converges to stationarity.

We build every concept from scratch:

| Section | What we build |
|---|---|
| §1 | **Matrices of a graph** — adjacency, degree, Laplacian |
| §2 | **Random walk as a Markov chain** — what it is, how it evolves |
| §3 | **Stationary distribution** — where does the walk end up? |
| §4 | **Detailed balance** — why the walk is reversible |
| §5 | **Spectral decomposition** — how eigenvalues control $P^t$ |
| §6 | **Spectral gap** — the one number that determines mixing speed |
| §7 | **Mixing time bound** — a rigorous inequality |
| §8 | **Analytical eigenvalues of the ring** — exact formulas |
| §9 | **Cheeger inequality** — spectral gap meets graph geometry |
| §10 | **$\beta$-sweep** — how all three networks compare spectrally |
| §11 | **Analytical $C(\beta)$** — derivation and verification |

**No prior knowledge assumed.** Each concept starts with a small concrete example.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.linalg import eigh

from src.smallworld.networks import build_ring, build_er, build_ws, build_all

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
COLORS = {'ring': '#4C72B0', 'ws': '#DD8452', 'er': '#55A868'}

---
## §1 — Matrices of a Graph

### 1.1 The adjacency matrix $A$

For a graph $G$ with $N$ nodes labelled $0,1,\ldots,N-1$:
$$A_{ij} = \begin{cases}1 & \text{if there is an edge between }i\text{ and }j,\\ 0 & \text{otherwise.}\end{cases}$$
$A$ is **symmetric** since the graph is undirected: $A_{ij}=A_{ji}$.

### 1.2 The degree matrix $D$

The **degree** $d_i = \sum_j A_{ij}$ counts the neighbours of node $i$.
$$D = \operatorname{diag}(d_0, d_1, \ldots, d_{N-1}).$$

### 1.3 The Laplacian $L = D - A$

$$L_{ij} = \begin{cases}d_i & i = j,\\ -1 & (i,j)\in E,\\ 0 & \text{otherwise.}\end{cases}$$
$L$ is symmetric and every row sums to zero (diagonal cancels the off-diagonal $-1$s).

### 1.4 The normalised Laplacian $\mathcal{L}$

$$\mathcal{L} = D^{-1/2}\,L\,D^{-1/2} = I - D^{-1/2}A\,D^{-1/2}.$$

For a $k$-regular graph ($D = kI$): $\mathcal{L} = I - A/k$.
Both the ring and WS are $k$-regular (WS preserves degrees exactly),
so this simplification applies throughout.

In [ ]:
# Small example: path graph P_4  (0 -- 1 -- 2 -- 3)
G_ex = nx.path_graph(4)
nodes = sorted(G_ex.nodes())
N_ex  = len(nodes)
idx   = {v: i for i, v in enumerate(nodes)}

A_ex = np.zeros((N_ex, N_ex))
for u, v in G_ex.edges():
    A_ex[idx[u], idx[v]] = A_ex[idx[v], idx[u]] = 1.0

deg_ex   = A_ex.sum(axis=1)
L_ex     = np.diag(deg_ex) - A_ex
Lnorm_ex = np.eye(N_ex) - np.diag(deg_ex**-0.5) @ A_ex @ np.diag(deg_ex**-0.5)

print('Path P_4 degrees:', deg_ex.astype(int))
print('\nA =\n', A_ex.astype(int))
print('\nL = D - A =\n', L_ex.astype(int))
print('\nRow sums of L:', L_ex.sum(axis=1))
print('\nNormalised Laplacian L_norm:\n', np.round(Lnorm_ex, 3))

### 1.5 Key properties of $\mathcal{L}$

**Property 1: $\mathcal{L}$ is symmetric positive semi-definite** (all eigenvalues real, $\geq 0$).

*Proof.* Let $g = D^{-1/2}f$. Then:
$$f^\top \mathcal{L}\, f = g^\top L g = \sum_{(i,j)\in E}(g_i - g_j)^2 \geq 0.$$
This is a sum of squares — always non-negative. It equals zero only when $g$
is constant on each connected component.

**Property 2: The smallest eigenvalue is $\mu_1 = 0$.**

Set $f = D^{1/2}\mathbf{1}$, so $g = \mathbf{1}$ (constant). Then $f^\top\mathcal{L}f = 0$.

**Property 3: $\mu_2 > 0$ iff $G$ is connected** (Fiedler, 1973).

If disconnected, a piecewise-constant $g$ also gives zero. If connected,
$\mathbf{1}$ is the *unique* zero, so $\mu_2 > 0$.
$\mu_2$ is called the **algebraic connectivity** or **Fiedler value**.

**Property 4:** All eigenvalues lie in $[0, 2]$.

In [ ]:
mu_ex, vecs_ex = eigh(Lnorm_ex)   # sorted ascending

print('Eigenvalues of L_norm for P_4:', np.round(mu_ex, 4))
print(f'  mu_1 = {mu_ex[0]:.6f}  (should be 0)')
print(f'  mu_2 = {mu_ex[1]:.6f}  (> 0 since P_4 is connected)')
print(f'  mu_4 = {mu_ex[-1]:.6f}  (<= 2)')
print()
print('First eigenvector (mu_1=0): proportional to sqrt(d_i)')
print(' computed:', np.round(vecs_ex[:, 0], 4))
print(' expected:', np.round(np.sqrt(deg_ex)/np.linalg.norm(np.sqrt(deg_ex)), 4))

---
## §2 — The Random Walk as a Markov Chain

### 2.1 Definition

At each time step, the walker at node $X_t$ moves to a uniformly random neighbour:
$$\Pr[X_{t+1} = j \mid X_t = i] = \frac{A_{ij}}{d_i}.$$

### 2.2 The transition matrix $P = D^{-1}A$

$P_{ij} = A_{ij}/d_i$. Two key properties:
- **Row sums are 1**: $\sum_j P_{ij} = 1$ (row-stochastic matrix).
- **All entries are non-negative**.

This is a **Markov chain**: the next state depends only on the current state,
not on the history.

### 2.3 Evolving a distribution

If $p^{(0)}$ is a probability distribution over nodes, after $t$ steps:
$$p^{(t)} = p^{(0)} P^t.$$

**Main question:** what is the limit of $p^{(t)}$ as $t \to \infty$?

In [ ]:
P_ex = A_ex / deg_ex[:, None]   # row i divided by d_i

print('Transition matrix P (P_ij = prob of going i -> j):')
print(np.round(P_ex, 3))
print('Row sums:', P_ex.sum(axis=1))
print()
print('Distribution p(t) starting at node 0:')
for t in [0, 1, 2, 5, 10, 20, 50, 100]:
    p_t = np.array([1., 0., 0., 0.]) @ np.linalg.matrix_power(P_ex, t)
    print(f'  t={t:3d}: {np.round(p_t, 4)}')

---
## §3 — The Stationary Distribution

### 3.1 Definition

A distribution $\pi$ is **stationary** if one more step leaves it unchanged:
$$\pi P = \pi.$$

Looking at the output above: $p^{(t)}$ converges to a fixed limit as $t$ grows.
Notice the walker spends more time at middle nodes (degree 2) than at endpoints (degree 1).
The walker accumulates at high-degree nodes.

### 3.2 Claim: $\pi_i = d_i / (2m)$ is stationary

**Proof.** We verify $(\pi P)_j = \pi_j$ for all $j$:
$$(\pi P)_j = \sum_i \pi_i P_{ij} = \sum_i \frac{d_i}{2m}\cdot\frac{A_{ij}}{d_i}
= \frac{1}{2m}\sum_i A_{ij} = \frac{d_j}{2m} = \pi_j. \quad\square$$

**Corollary.** For $k$-regular graphs (ring, WS), $\pi_i = 1/N$ — **uniform**.
The walker visits every node equally often in the long run.

### 3.3 Time-average interpretation

By the ergodic theorem, $\pi_i$ is the long-run fraction of time spent at node $i$:
$$\pi_i = \lim_{T\to\infty}\frac{1}{T}\#\{t\leq T : X_t = i\}.$$
This is what we measured in notebook 04.

In [ ]:
def stationary_distribution(G):
    nodes = sorted(G.nodes())
    deg = np.array([G.degree(v) for v in nodes], dtype=float)
    return deg / deg.sum()


# Check on P_4 (degrees [1,2,2,1], sum=6)
pi_ex = deg_ex / deg_ex.sum()
print('P_4  degrees:    ', deg_ex.astype(int))
print('     pi (d/2m):  ', np.round(pi_ex, 4))
print('     pi P - pi:  ', np.round(pi_ex @ P_ex - pi_ex, 10))
print()

# For an irregular WS graph (some degrees differ after rewiring)
G_ws_test = build_ws(30, 4, 0.3, seed=7)
pi_ws = stationary_distribution(G_ws_test)
nodes_ws = sorted(G_ws_test.nodes())
idx_ws = {v: i for i, v in enumerate(nodes_ws)}
N_ws = len(nodes_ws)
A_ws = np.zeros((N_ws, N_ws))
for u, v in G_ws_test.edges():
    A_ws[idx_ws[u], idx_ws[v]] = A_ws[idx_ws[v], idx_ws[u]] = 1.0
deg_ws = A_ws.sum(axis=1)
P_ws = A_ws / deg_ws[:, None]
print(f'WS(30,4,0.3):  max|pi P - pi| = {np.max(np.abs(pi_ws @ P_ws - pi_ws)):.2e}  (~0)')

---
## §4 — Detailed Balance and Reversibility

### 4.1 Definition

Imagine watching the walk at stationarity on video.
**Detailed balance** says the video is indistinguishable forwards and backwards:
$$\pi_i P_{ij} = \pi_j P_{ji} \quad \text{for all } i, j.$$

Left side: (probability of being at $i$) $\times$ (probability of stepping to $j$).
Right side: same for the reverse direction $j \to i$.

### 4.2 Our walk satisfies detailed balance

**Proof.** Using $\pi_i = d_i/(2m)$ and $P_{ij} = A_{ij}/d_i$:
$$\pi_i P_{ij} = \frac{d_i}{2m}\cdot\frac{A_{ij}}{d_i} = \frac{A_{ij}}{2m}.$$
By symmetry $A_{ij}=A_{ji}$:
$$\pi_j P_{ji} = \frac{A_{ji}}{2m} = \frac{A_{ij}}{2m}. \quad\square$$

Both sides equal $A_{ij}/(2m)$ — the probability of edge $(i,j)$ being traversed.

### 4.3 The symmetrisation trick

A chain satisfying detailed balance is **reversible**.
The conjugated matrix $\hat{P} = D^{1/2}P\,D^{-1/2}$ is symmetric:
$$\hat{P}_{ij} = \frac{A_{ij}}{\sqrt{d_i d_j}} = \hat{P}_{ji}.$$

Expanding: $\hat{P} = D^{-1/2}A\,D^{-1/2} = I - \mathcal{L}$.

So $P$ and $\mathcal{L}$ are **similar matrices** (related by conjugation with $D^{1/2}$):
if $\mathcal{L}\psi = \mu\psi$ then $P\phi = (1-\mu)\phi$ where $\phi = D^{-1/2}\psi$.

**Key link: eigenvalues of $\mathcal{L}$ directly give eigenvalues of $P$.**

In [ ]:
# Verify detailed balance on P_4
balance = pi_ex[:, None] * P_ex   # M_ij = pi_i * P_ij
print('pi_i * P_ij (should equal its own transpose):')
print(np.round(balance, 4))
print('Max |M - M^T| =', np.max(np.abs(balance - balance.T)))
print()

# Show P_hat = D^(1/2) P D^(-1/2) = I - L_norm
D_sq      = np.diag(np.sqrt(deg_ex))
P_hat     = D_sq @ P_ex @ np.diag(1./np.sqrt(deg_ex))
I_minus_L = np.eye(N_ex) - Lnorm_ex
print('P_hat = D^(1/2) P D^(-1/2):')
print(np.round(P_hat, 4))
print('\nI - L_norm:')
print(np.round(I_minus_L, 4))
print('\nMax |P_hat - (I - L_norm)| =', np.max(np.abs(P_hat - I_minus_L)))

# Eigenvalue correspondence: rho_j = 1 - mu_j
rho = np.sort(np.linalg.eigvals(P_ex).real)[::-1]
print('\nEigenvalues of P:       ', np.round(rho, 4))
print('1 - eigenvalues of L:   ', np.round(1 - mu_ex, 4))

---
## §5 — Spectral Decomposition of $P^t$

### 5.1 Diagonalising the walk

Since $\hat{P} = I - \mathcal{L}$ is symmetric, the spectral theorem gives
an orthonormal eigenbasis $\{\psi_1,\ldots,\psi_N\}$ with eigenvalues
$\rho_j = 1-\mu_j$, ordered $1 = \rho_1 \geq \rho_2 \geq \cdots \geq \rho_N$.

In this basis, $\hat{P}^t$ multiplies eigenvector $\psi_j$ by $\rho_j^t$.
Converting back, the distribution after $t$ steps from node $x$ is:
$$p_x^{(t)}(y) = \pi(y)\!\left[1 + \sum_{j=2}^N \rho_j^t\,
\frac{\psi_j(x)\psi_j(y)}{\sqrt{\pi(x)\pi(y)}}\right].$$

- The $j=1$ term gives exactly $\pi(y)$.
- The sum for $j \geq 2$ is the **error** — it decays like $\rho_j^t$.

### 5.2 Perron-Frobenius theorem

For a connected, non-bipartite graph:
- $\rho_1 = 1$ is the **unique** largest eigenvalue.
- All other $|\rho_j| < 1$.

So the error vanishes: **the walk always converges to $\pi$ regardless of start.**

### 5.3 Rate of convergence

The rate is controlled by $\rho^* = \max_{j\geq 2}|\rho_j| = 1-\lambda^*$,
where $\lambda^* = \mu_2$ is the **spectral gap**.

The error decays like $(\rho^*)^t = (1-\lambda^*)^t$.

**Maximising $\lambda^*$ $\Leftrightarrow$ minimising $\rho^*$ $\Leftrightarrow$ fastest convergence.**

In [ ]:
# Visualise convergence from node 0: ring vs WS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label, G_conv) in zip(axes, [
    ('Ring C(20,4)',  build_ring(20, 4)),
    ('WS (beta=0.1)', build_ws(20, 4, 0.1, seed=0, connected=True)),
]):
    nodes_c = sorted(G_conv.nodes())
    N_c = len(nodes_c)
    idx_c = {v: i for i, v in enumerate(nodes_c)}
    A_c = np.zeros((N_c, N_c))
    for u, v in G_conv.edges():
        A_c[idx_c[u], idx_c[v]] = A_c[idx_c[v], idx_c[u]] = 1.0
    deg_c  = A_c.sum(axis=1)
    P_c    = A_c / deg_c[:, None]
    pi_c   = deg_c / deg_c.sum()

    Lnorm_c = np.eye(N_c) - np.diag(deg_c**-0.5) @ A_c @ np.diag(deg_c**-0.5)
    lam_c = float(eigh(Lnorm_c, eigvals_only=True, subset_by_index=[1, 1]))

    p_curr = np.zeros(N_c); p_curr[0] = 1.0
    tv_dists, ts = [], list(range(201))
    for t in ts:
        tv_dists.append(0.5 * np.sum(np.abs(p_curr - pi_c)))
        p_curr = p_curr @ P_c

    bound = [0.5 * (1 - lam_c)**t for t in ts]
    ax.semilogy(ts, tv_dists, color='steelblue', lw=1.5, label='TV distance')
    ax.semilogy(ts, bound, color='orange', lw=1.2, linestyle=':', label='Bound (1-lam*)^t / 2')
    ax.axhline(0.25, color='red', lw=1, linestyle='--', alpha=0.6, label='eps=0.25')
    ax.set_title(f'{label}  (lam* = {lam_c:.4f})', fontsize=10)
    ax.set_xlabel('Steps t'); ax.set_ylabel('TV distance from pi')
    ax.legend(fontsize=8); ax.set_xlim(0, 200)

fig.suptitle('Convergence of the random walk to stationarity', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / '05_convergence.png', dpi=150)
plt.show()

---
## §6 — The Spectral Gap

### 6.1 Definition

Order eigenvalues of $\mathcal{L}$: $0 = \mu_1 \leq \mu_2 \leq \cdots \leq \mu_N$.
$$\lambda^* = \mu_2 \quad \text{(the spectral gap — smallest non-zero eigenvalue).}$$

### 6.2 Variational characterisation (Courant-Fischer)

$$\lambda^* = \min_{\substack{g:\,\sum_i\pi_i g_i = 0 \\ g\neq 0}}
\frac{\sum_{(i,j)\in E}(g_i-g_j)^2}{\sum_i\pi_i g_i^2}.$$

Numerator: total variation of $g$ across edges.
Denominator: overall variance of $g$.

**A small $\lambda^*$ means there is a smooth function $g$ (varies little across edges)
that is not constant — only possible if the graph has a bottleneck.**

### 6.3 Intuition for our three networks

- **Ring**: Set $g=+1$ on the left half, $g=-1$ on the right. Only $k$ edges cross
  the cut, so the numerator is $O(k)$ while the denominator is $O(N)$.
  This gives $\lambda^* \sim N^{-2}$ — very small.

- **WS**: Each rewired edge is a shortcut across the cut. Even a few force $g$ to jump,
  raising the numerator and hence $\lambda^*$.

- **ER**: Many random edges; no bottleneck. $\lambda^* = O(1)$.

In [ ]:
def build_normalised_laplacian(G):
    nodes = sorted(G.nodes())
    N = len(nodes)
    idx = {v: i for i, v in enumerate(nodes)}
    A = np.zeros((N, N))
    for u, v in G.edges():
        A[idx[u], idx[v]] = A[idx[v], idx[u]] = 1.0
    deg = A.sum(axis=1)
    return np.eye(N) - np.diag(deg**-0.5) @ A @ np.diag(deg**-0.5)


def spectral_gap(G):
    # mu_2 = smallest non-zero eigenvalue of the normalised Laplacian
    Lnorm = build_normalised_laplacian(G)
    mu = eigh(Lnorm, eigvals_only=True, subset_by_index=[0, 1])
    return float(mu[1])


N, k = 100, 6
print(f'Spectral gap lambda* for N={N}, k={k}:')
for name, G in [('ring',      build_ring(N, k)),
                ('ws b=0.05', build_ws(N, k, 0.05, seed=0, connected=True)),
                ('er',        build_er(N, k, seed=0))]:
    lam = spectral_gap(G)
    print(f'  {name:>12}:  lambda* = {lam:.5f}   rho* = {1-lam:.5f}')

---
## §7 — The Mixing-Time Bound

### 7.1 Total variation distance

$$\|p - q\|_{\mathrm{TV}} = \frac{1}{2}\sum_{x\in V}|p(x) - q(x)|.$$

This is the worst-case difference in probability over all subsets of nodes.
When TV distance is small, $p$ and $q$ are nearly indistinguishable.

### 7.2 Main theorem

**Theorem.** Starting from node $x$:
$$\left\|p_x^{(t)} - \pi\right\|_{\mathrm{TV}}
\leq \frac{1}{2}\sqrt{\frac{2m}{d_x}}\,(1-\lambda^*)^t.$$

**Proof sketch.** From the spectral decomposition, apply Cauchy-Schwarz:
$$\|p_x^{(t)}-\pi\|_{\mathrm{TV}}^2
\leq \frac{(1-\lambda^*)^{2t}}{4\pi(x)}\underbrace{\sum_{j\geq 2}\psi_j(x)^2}_{\leq 1}. \quad\square$$

### 7.3 Mixing time

$$t_{\mathrm{mix}}(\varepsilon)
= \min\!\left\{t : \max_{x}\|p_x^{(t)}-\pi\|_{\mathrm{TV}}\leq\varepsilon\right\}.$$

Setting the bound equal to $\varepsilon$ and solving for $t$:
$$\boxed{t_{\mathrm{mix}}(\varepsilon) \leq
\frac{1}{\lambda^*}\ln\!\left(\frac{\sqrt{d_{\max}/d_{\min}}}{2\varepsilon}\right)}.$$

For $k$-regular graphs: $t_{\mathrm{mix}} \leq \lambda^{*-1}\ln(1/(2\varepsilon))$.

**Mixing time is $\Theta(1/\lambda^*)$. Double the spectral gap, halve the mixing time.**

In [ ]:
def mixing_time_bound(G, eps=0.25):
    lam = spectral_gap(G)
    if lam <= 0: return float('inf')
    degrees = [d for _, d in G.degree()]
    return (1./lam) * np.log(np.sqrt(max(degrees)/min(degrees)) / (2*eps))


def mixing_time_sim(G, eps=0.25, n_starts=10, max_t=5000):
    # Evolve p^(t) exactly from n_starts nodes, find first t with TV <= eps
    nodes = sorted(G.nodes())
    N = len(nodes)
    idx = {v: i for i, v in enumerate(nodes)}
    A = np.zeros((N, N))
    for u, v in G.edges():
        A[idx[u], idx[v]] = A[idx[v], idx[u]] = 1.0
    deg = A.sum(axis=1)
    P = A / deg[:, None]
    pi = deg / deg.sum()
    rng = np.random.default_rng(0)
    starts = rng.choice(N, size=min(n_starts, N), replace=False)
    worst = 0
    for x in starts:
        p = np.zeros(N); p[x] = 1.0
        for t in range(1, max_t+1):
            p = p @ P
            if 0.5*np.sum(np.abs(p-pi)) <= eps:
                worst = max(worst, t); break
        else:
            worst = max_t
    return worst


N, k = 100, 6
configs = [
    ('ring',       build_ring(N, k)),
    ('ws b=0.001', build_ws(N, k, 0.001, seed=0, connected=True)),
    ('ws b=0.01',  build_ws(N, k, 0.01,  seed=0, connected=True)),
    ('ws b=0.05',  build_ws(N, k, 0.05,  seed=0, connected=True)),
    ('ws b=0.1',   build_ws(N, k, 0.1,   seed=0, connected=True)),
    ('er',         build_er(N, k, seed=0)),
]

print(f'{"Network":>12}  {"lambda*":>8}  {"Bound":>8}  {"Simul.":>8}')
print('-' * 44)
for name, G in configs:
    lam, bound, sim = spectral_gap(G), mixing_time_bound(G), mixing_time_sim(G)
    print(f'{name:>12}  {lam:>8.5f}  {bound:>8.1f}  {sim:>8d}')

---
## §8 — Analytical Eigenvalues of the Ring Lattice

### 8.1 Circulant graphs and the DFT

The ring $C(N,k)$ is a **circulant graph** (full rotational symmetry):
node $i$ connects to $(i \pm s) \bmod N$ for $s=1,\ldots,k/2$.

For circulant graphs, the eigenvectors of $A$ are the **discrete Fourier modes**:
$$\omega^{(j)}_\ell = \frac{1}{\sqrt{N}}e^{2\pi i j\ell/N}, \quad j=0,\ldots,N-1.$$

Computing $(A\omega^{(j)})_\ell$ using the shift structure of the ring:
$$(A\omega^{(j)})_\ell
= \sum_{s=1}^{k/2}\!\left(\omega^{(j)}_{\ell+s} + \omega^{(j)}_{\ell-s}\right)
= \left[\sum_{s=1}^{k/2}2\cos\!\left(\frac{2\pi js}{N}\right)\right]\omega^{(j)}_\ell.$$

So the eigenvalue of $A$ for Fourier mode $j$ is:
$$\lambda^{(A)}_j = \sum_{s=1}^{k/2}2\cos\!\left(\frac{2\pi js}{N}\right).$$

Since the ring is $k$-regular, $\mathcal{L} = I - A/k$, so $\mu_j = 1 - \lambda^{(A)}_j/k$.
The spectral gap is the $j=1$ mode (slowest spatial variation):
$$\lambda^*_{\mathrm{ring}} = 1 - \frac{1}{k}\sum_{s=1}^{k/2}2\cos\!\left(\frac{2\pi s}{N}\right).$$

### 8.2 Asymptotic approximation for $N \gg k$

Using $\cos\theta \approx 1 - \theta^2/2$ for small $\theta = 2\pi s/N$:
$$\lambda^*_{\mathrm{ring}} \approx \frac{\pi^2 k^2}{12 N^2}.$$

This gives $t_{\mathrm{mix}} \sim N^2/k^2$: **quadratic in $N$**, the hallmark
of diffusion on a 1D periodic lattice.

In [ ]:
def ring_eigenvalues_analytical(N, k):
    # mu_j = 1 - (1/k) * sum_{s=1}^{k/2} 2*cos(2*pi*j*s/N)
    js = np.arange(N)
    ss = np.arange(1, k//2 + 1)
    lam_A = 2 * np.cos(2*np.pi*np.outer(js, ss)/N).sum(axis=1)
    return np.sort(1 - lam_A / k)


def ring_gap_approx(N, k):
    # Asymptotic: lambda* ~ pi^2 k^2 / (12 N^2)
    return np.pi**2 * np.asarray(k, float)**2 / (12 * np.asarray(N, float)**2)


k = 6
print(f'k = {k}')
print(f'{"N":>6}  {"Analytical":>12}  {"Numerical":>12}  {"Approx pi^2k^2/(12N^2)":>24}')
print('-' * 60)
for N in [20, 50, 100, 200, 500, 1000]:
    mu_an = ring_eigenvalues_analytical(N, k)[1]
    mu_nu = spectral_gap(build_ring(N, k))
    mu_ap = ring_gap_approx(N, k)
    print(f'{N:>6}  {mu_an:>12.7f}  {mu_nu:>12.7f}  {mu_ap:>24.7f}')

In [ ]:
# Log-log plot: lambda* vs N -- verify N^(-2) scaling
k = 6
Ns = np.array([20, 30, 50, 80, 100, 150, 200, 300, 500, 800, 1000])
gaps_exact  = np.array([ring_eigenvalues_analytical(N, k)[1] for N in Ns])
gaps_approx = ring_gap_approx(Ns, k)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(Ns, gaps_exact,  'o-', color=COLORS['ring'], label='Analytical mu_2')
ax.loglog(Ns, gaps_approx, '--', color='grey', label='pi^2 k^2 / (12 N^2)')
Nref = np.array([100, 1000])
yref = gaps_approx[Ns == 100][0] * (Nref/100)**(-2) * 2
ax.loglog(Nref, yref, 'k:', lw=1)
ax.text(280, yref[0]*0.35, 'slope -2', fontsize=9)
ax.set_xlabel('N'); ax.set_ylabel('lambda*')
ax.set_title(f'Ring spectral gap vs N  (k={k})')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '05_ring_spectral_gap_scaling.png', dpi=150)
plt.show()

print('lambda*(ring) ~ N^(-2)  =>  t_mix ~ N^2')
print(f'N=1000, k=6: lambda* ~ {ring_gap_approx(1000,6):.2e}')
print(f't_mix bound ~ {1/ring_gap_approx(1000,6)*np.log(1/0.5):.0f} steps')

---
## §9 — Cheeger Inequality: Spectral Gap Meets Geometry

### 9.1 Bottlenecks

A walk mixes slowly when the graph has a **bottleneck**: a small set of edges
separating the graph into two large pieces. Once on one side, crossing over takes many steps.

### 9.2 Edge conductance

For $S \subseteq V$:
- $\partial S$: edges crossing the cut $(S,\, V\setminus S)$.
- $\mathrm{vol}(S) = \sum_{i \in S} d_i$: total degree inside $S$.

$$\Phi(S) = \frac{|\partial S|}{\min(\mathrm{vol}(S),\,\mathrm{vol}(V\setminus S))},
\qquad h(G) = \min_{S}\Phi(S).$$

Small $h(G)$ means the graph has a bad bottleneck.

### 9.3 The Cheeger inequality (Alon-Milman, 1985)

$$\frac{h(G)^2}{2} \leq \lambda^* \leq 2\,h(G).$$

- **Upper bound**: a bad bottleneck forces a small $\lambda^*$.
- **Lower bound**: $\lambda^*$ is at least $h^2/2$ — the bottleneck is the only obstacle.

### 9.4 Ring lattice: $h_{\mathrm{ring}} = 2/N$

Worst cut: balanced half $S = \{0,\ldots,N/2-1\}$.
This has $k$ crossing edges and $\mathrm{vol}(S) = kN/2$:
$$h_{\mathrm{ring}} = \frac{k}{kN/2} = \frac{2}{N}.$$

Cheeger predicts $2/N^2 \leq \lambda^*_{\mathrm{ring}} \leq 4/N$,
consistent with our exact result $\approx \pi^2k^2/(12N^2)$.

### 9.5 Effect of rewiring

Each WS shortcut adds crossing edges to the worst-case cut, raising $h(G)$.
Via $\lambda^* \geq h(G)^2/2$, even a few shortcuts can dramatically increase $\lambda^*$.

In [ ]:
def cheeger_fiedler(G):
    # Approximate h(G) via the Fiedler vector sweep.
    # Sorts nodes by mu_2 eigenvector and tries all threshold cuts.
    # Returns (h_approx, best_S). Guaranteed: h_approx <= sqrt(2 * lambda*).
    Lnorm = build_normalised_laplacian(G)
    nodes = sorted(G.nodes())
    N = len(nodes)
    _, vecs = eigh(Lnorm, subset_by_index=[1, 1])
    fiedler = vecs[:, 0]
    deg = np.array([G.degree(v) for v in nodes], dtype=float)
    vol_total = deg.sum()
    sorted_nodes = [nodes[i] for i in np.argsort(fiedler)]

    h_min, best_S, vol_S, S_set, cut = float('inf'), set(), 0., set(), 0
    for node in sorted_nodes[:-1]:
        S_set.add(node)
        vol_S += G.degree(node)
        for nb in G.neighbors(node):
            cut += 1 if nb not in S_set else -1
        denom = min(vol_S, vol_total - vol_S)
        if denom > 0 and cut/denom < h_min:
            h_min = cut/denom; best_S = S_set.copy()
    return h_min, best_S


N, k = 80, 6
print(f'Cheeger inequality verification  (N={N}, k={k})')
print(f'  Theoretical h_ring = 2/N = {2/N:.5f}\n')
print(f'{"Network":>14}  {"lambda*":>8}  {"h(Fiedler)":>12}  {"h^2/2":>8}  {"ok":>5}  {"2h":>8}  {"ok":>5}')
print('-' * 65)
for name, G in [('ring',      build_ring(N, k)),
                ('ws b=0.01', build_ws(N, k, 0.01, seed=0, connected=True)),
                ('ws b=0.1',  build_ws(N, k, 0.1,  seed=0, connected=True)),
                ('er',        build_er(N, k, seed=0))]:
    lam = spectral_gap(G)
    h, _ = cheeger_fiedler(G)
    lo, hi = h**2/2, 2*h
    print(f'{name:>14}  {lam:>8.5f}  {h:>12.5f}  {lo:>8.5f}  '
          f'{"OK" if lam>=lo-1e-9 else "FAIL":>5}  {hi:>8.5f}  '
          f'{"OK" if lam<=hi+1e-9 else "FAIL":>5}')

---
## §10 — $\beta$-sweep of the Spectral Gap

We add $\lambda^*(\beta)$ to the classic small-world plot.

**Expected pattern:**
- $L(\beta)/L(0)$: drops quickly (few shortcuts shrink the diameter).
- $C(\beta)/C(0)$: drops slowly (most triangles survive).
- $\lambda^*(\beta)/\lambda^*_{\mathrm{ER}}$: rises quickly, mirroring $L$
  (the same shortcuts control both path length and mixing speed).

The **small-world window** ($\beta \approx 0.01$-$0.1$) is where
$L$ has fallen and $\lambda^*$ has risen, but $C$ is still high.

In [ ]:
# Hand-built metrics (no nx analysis helpers -- professor's rule)

def bfs_distances(G, source):
    dist, queue, head = {source: 0}, [source], 0
    while head < len(queue):
        u = queue[head]; head += 1
        for v in G.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1; queue.append(v)
    return dist

def avg_path_length(G):
    total, count = 0, 0
    for u in G.nodes():
        for d in bfs_distances(G, u).values():
            if d > 0: total += d; count += 1
    return total/count if count > 0 else 0.

def clustering_coefficient(G):
    triangles, paths2 = 0, 0
    for v in G.nodes():
        nbrs = list(G.neighbors(v))
        dv = len(nbrs)
        paths2 += dv*(dv-1)//2
        for i in range(len(nbrs)):
            for j in range(i+1, len(nbrs)):
                if G.has_edge(nbrs[i], nbrs[j]): triangles += 1
    return triangles/paths2 if paths2 > 0 else 0.


N, k, n_trials = 150, 6, 5
betas = [0.0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
results = []
for beta in betas:
    Ls, Cs, lams = [], [], []
    for trial in range(n_trials):
        if beta == 0.0:   G = build_ring(N, k)
        elif beta == 1.0: G = build_er(N, k, seed=trial)
        else:             G = build_ws(N, k, beta, seed=trial, connected=True)
        Ls.append(avg_path_length(G))
        Cs.append(clustering_coefficient(G))
        lams.append(spectral_gap(G))
    results.append({'beta':beta, 'L':np.mean(Ls), 'C':np.mean(Cs), 'lam':np.mean(lams)})
    print(f"beta={beta:.3f}  L={results[-1]['L']:.2f}  C={results[-1]['C']:.3f}  lam={results[-1]['lam']:.5f}")

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

L0   = df.loc[df['beta']==0.0, 'L'].values[0]
C0   = df.loc[df['beta']==0.0, 'C'].values[0]
lam1 = df.loc[df['beta']==1.0, 'lam'].values[0]

df['L_norm']   = df['L']  / L0
df['C_norm']   = df['C']  / C0
df['lam_norm'] = df['lam']/ lam1
df_plot = df[df['beta'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(df_plot['beta'], df_plot['L_norm'],   'o-',  color='#4C72B0', lw=2, label='L(beta)/L(0)')
ax.semilogx(df_plot['beta'], df_plot['C_norm'],   's-',  color='#DD8452', lw=2, label='C(beta)/C(0)')
ax.semilogx(df_plot['beta'], df_plot['lam_norm'], '^--', color='#55A868', lw=2, label='lambda*(beta)/lambda*_ER')
ax.axvspan(0.01, 0.1, alpha=0.13, color='gold', label='Small-world window')
ax.axhline(1., color='grey', lw=0.8, linestyle=':')
ax.set_xlabel('Rewiring probability beta', fontsize=12)
ax.set_ylabel('Normalised value', fontsize=12)
ax.set_title(f'N={N}, k={k}: path length, clustering, spectral gap vs beta', fontsize=11)
ax.legend(fontsize=9, loc='center left'); ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.savefig(FIGURES / '05_beta_sweep_spectral.png', dpi=150)
plt.show()

print(df_plot[['beta','L_norm','C_norm','lam_norm']].to_string(
    index=False, float_format='{:.4f}'.format))

---
## §11 — Analytical Approximation $C(\beta) \approx C(0)(1-\beta)^3$

### 11.1 Exact $C(0)$ for the ring

Recall $C = 3 \times \text{(triangles)} / \text{(paths of length 2)}$.

**Paths of length 2.** Each node has degree $k$, giving $\binom{k}{2} = k(k-1)/2$
paths centred on it. Total: $Nk(k-1)/2$.

**Triangles.** Fix node $v$. Neighbours: $\{v \pm 1, \ldots, v \pm k/2\} \pmod{N}$.
Two neighbours $v+s$ and $v+t$ form a triangle with $v$ iff they are adjacent,
i.e. iff $|s-t| \leq k/2$ (they are close enough on the ring to see each other).
Counting all valid pairs for large $N$:
$$C(0) = \frac{3(k-2)}{4(k-1)}.$$

Check: $k=6 \Rightarrow C(0) = 0.6$; $\quad k=4 \Rightarrow C(0) = 0.5$.

### 11.2 Derivation of $(1-\beta)^3$

A triangle $(u,v,w)$ in the original ring survives the WS rewiring iff
**all three of its edges survive** (none is rewired away).
Each edge is rewired independently with probability $\beta$:
$$\Pr[\text{triangle survives}] = (1-\beta)^3.$$

The WS process preserves degrees exactly, so the number of length-2 paths
is unchanged. Therefore:
$$\boxed{C(\beta) \approx C(0) \cdot (1-\beta)^3.}$$

Corrections are $O(\beta^2)$ from new triangles created by rewired edges.

In [ ]:
def C0_ring(k):
    # Exact clustering of C(N,k) for large N: 3(k-2) / (4(k-1))
    return 3*(k-2) / (4*(k-1))


def C_ws_analytical(beta, k):
    # WS approximation: C(beta) ~ C(0) * (1-beta)^3
    return C0_ring(k) * (1 - np.asarray(beta))**3


N, k = 400, 6
print(f'C(0) formula:   {C0_ring(k):.4f}')
print(f'C(0) measured:  {clustering_coefficient(build_ring(N, k)):.4f}')
print()
print(f'{"beta":>8}  {"Formula":>10}  {"Measured":>12}  {"Rel. error":>12}')
print('-' * 50)
for beta in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]:
    c_f = C_ws_analytical(beta, k)
    c_m = np.mean([clustering_coefficient(build_ws(N, k, beta, seed=t, connected=True))
                   for t in range(8)])
    print(f'{beta:>8.3f}  {c_f:>10.4f}  {c_m:>12.4f}  {abs(c_f-c_m)/c_m*100:>11.1f}%')

In [ ]:
N, k = 400, 6
betas_dense = np.logspace(-3, 0, 50)

C_sim = [np.mean([clustering_coefficient(build_ws(N, k, b, seed=t, connected=True))
                  for t in range(6)]) for b in betas_dense]
C_analytic = C_ws_analytical(betas_dense, k)
C0 = C0_ring(k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogx(betas_dense, C_sim, 'o', color=COLORS['ws'], markersize=3, label='Simulation')
axes[0].semilogx(betas_dense, C_analytic, '-', color='black', lw=2, label='C(0)(1-beta)^3')
axes[0].axhline(C0, color='grey', lw=0.8, linestyle=':', label=f'C(0)={C0:.2f}')
axes[0].set_xlabel('beta (log scale)'); axes[0].set_ylabel('C(beta)')
axes[0].set_title('Clustering vs beta'); axes[0].legend(fontsize=9)

axes[1].loglog(betas_dense, np.array(C_sim)/C0, 'o',
               color=COLORS['ws'], markersize=3, label='Simulation / C(0)')
axes[1].loglog(betas_dense, C_analytic/C0, '-',
               color='black', lw=2, label='(1-beta)^3')
axes[1].set_xlabel('beta'); axes[1].set_ylabel('C(beta) / C(0)')
axes[1].set_title('Normalised clustering (log-log)'); axes[1].legend(fontsize=9)

fig.suptitle(f'C(beta) ~ C(0)(1-beta)^3  --  N={N}, k={k}', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / '05_clustering_analytical.png', dpi=150)
plt.show()

---
## Summary

### What we built, step by step

| Concept | Key result |
|---|---|
| Laplacian | $\mathcal{L} = D^{-1/2}(D-A)D^{-1/2}$, PSD, spectrum in $[0,2]$ |
| Stationary dist. | $\pi_i = d_i/(2m)$; uniform for regular graphs |
| Detailed balance | $\pi_i P_{ij} = \pi_j P_{ji}$; implies $\hat{P} = I - \mathcal{L}$ |
| Spectral decomp. | $p^{(t)}$ converges at rate $\rho^* = 1-\lambda^*$ |
| Mixing time bound | $t_{\rm mix} \leq (1/\lambda^*)\ln(\cdots)$ |
| Ring spectral gap | $\lambda^*_{\rm ring} \approx \pi^2k^2/(12N^2)$, so $t_{\rm mix} \sim N^2$ |
| Cheeger inequality | $h(G)^2/2 \leq \lambda^* \leq 2h(G)$ |
| WS shortcuts | Raise $h(G)$ and $\lambda^*$ from $O(N^{-2})$ to $O(N^{-1})$ or better |
| $C(\beta)$ formula | $C(\beta) \approx C(0)(1-\beta)^3$, $C(0)=3(k-2)/(4(k-1))$ |

### The big picture

| Network | $\lambda^*$ | $t_{\rm mix}$ | $C$ |
|---|---|---|---|
| Ring ($\beta=0$) | $\sim N^{-2}$ | $\sim N^2$ | High |
| WS ($\beta \approx 0.05$) | $\sim N^{-1}$ to $O(1)$ | $O(N)$ to $O(\log N)$ | Still high |
| ER ($\beta=1$) | $O(1)$ | $O(\log N)$ | Low |

Each rewired edge adds a shortcut that crosses the worst-case bottleneck,
raising $h(G)$, which via $\lambda^* \geq h(G)^2/2$ raises the spectral gap,
which via the mixing-time bound cuts $t_{\rm mix}$.
This is the rigorous explanation of why small-world networks mix so fast.